# Burgers Equation: Function Space vs Weight Space

This notebook conditions a space-time GP on the viscous **Burgers PDE**

$$u_t + u\,u_x = \nu\,u_{xx}, \qquad \nu = 0.02,$$

with zero Dirichlet boundaries, starting from a GP posterior given a few noisy observations of a sine initial condition. It does so two ways: a **function-space** flow over the $N = H \times W$ grid values, with the residual formed by finite differences, and an **RFF weight-space** flow over $R$ random-Fourier-feature weights, whose basis functions have analytic derivatives. The last section shows how the wall time of each scales with $N$.

## 1. Imports

In [ ]:
import sys
import time
from pathlib import Path

import gpytorch
import matplotlib.pyplot as plt
import torch

from flow_inference.conditions import EqualityCondition, PointWiseCondition, ProductCondition
from flow_inference.flows import BasisGaussianProcessFlow, GaussianProcessFlow
from flow_inference.gp_utils import DecoupledRFFBasis, gp_posterior
from flow_inference.guidance import GuidanceMC
from flow_inference.sampler import IntegratorSampler

sys.path.insert(0, str(Path.cwd().parent))  # examples/, for plotting.py
from plotting import show_fields

torch.set_default_dtype(torch.float64)

## 2. Configuration

The unit grid maps to space $x \in [-1, 1]$ along $H$ and time $t \in [0, 1)$ along $W$. The space axis includes both endpoints, so the Dirichlet conditions can be pinned at $x = \pm 1$ exactly; the spacing is $\Delta x = 2 / (H - 1)$.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
H = 50               # space points
W = 20               # time points
dim = 2              # inputs (x, t)
num_data = 5         # initial-condition observations
noise = 0.01**2      # observation noise variance
R = 256              # random Fourier features
num_samples = 5
SEED = 0

torch.manual_seed(SEED)  # reproducible runs


def build_grid(H, W, device):
    """(H*W, 2) unit-box coordinates; space includes both ends, so x = 2*c - 1 spans [-1, 1]."""
    xs = torch.linspace(0.0, 1.0, H, device=device)
    ts = torch.arange(W, device=device) / W
    xx, tt = torch.meshgrid(xs, ts, indexing="ij")
    return torch.stack([xx, tt], dim=-1).view(-1, 2)

## 3. Function-Space Flow

The GP has an anisotropic RBF kernel, set by hand, conditioned on noisy observations of $u(x, 0) = -\sin(\pi x)$. That posterior is the flow's starting distribution, before any PDE conditioning.

In [ ]:
x_initial = torch.tensor([i / num_data for i in range(num_data)]).to(device)  # grid coordinate of x
Y_initial = -torch.sin(torch.pi * (2 * x_initial - 1.0))
Y_initial = Y_initial + noise**0.5 * torch.randn_like(Y_initial)  # noise is a variance
X_initial = torch.stack([x_initial, torch.zeros_like(x_initial)], dim=1)  # (num_data, 2), at t = 0

coords = build_grid(H, W, device)  # (H*W, 2)
kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel(ard_num_dims=2)).to(device)
kernel.base_kernel.lengthscale = torch.tensor([0.025, 0.3]).to(device)
kernel.outputscale = 1.0
p0_mean_fn, p0_kernel = gp_posterior(kernel, X_initial, Y_initial, noise=noise)

Samples given the initial condition only: no physics, no boundaries.

In [ ]:
flow = GaussianProcessFlow(p0_kernel, p0_mean_fn, coords, device=device)
sampler = IntegratorSampler(flow=flow)
uncond_samples = sampler.sample(num_samples, guidance=None, num_steps=1_000)
show_fields(uncond_samples.view(-1, H, W)[:5], cmap="bwr")

`BurgersCondition` evaluates the PDE residual by centred differences at the interior grid points, and a `PointWiseCondition` pins the first and last spatial rows to zero at all times. `ProductCondition` imposes both.

In [ ]:
class BurgersCondition(EqualityCondition):
    """Burgers residual by finite differences, for a single-channel field in (H, W) order."""

    def __init__(self, H, W, sigma=10.0):
        super().__init__(sigma=sigma)
        self.H = H
        self.W = W
        self.dx = 2.0 / (H - 1)  # the grid includes x = -1 and x = 1
        self.dt = 1.0 / W
        self.nu = 0.02  # viscosity

    def residual(self, x):  # x: (B, 1, H*W)
        B, C, N = x.shape
        assert C == 1, f"Expected C=1, got C={C}"
        assert N == self.H * self.W, f"Expected N={self.H*self.W}, got N={N}"

        u = x.view(B, self.H, self.W)
        u_int = u[:, 1:-1, 1:-1]  # interior points, (B, H-2, W-2)
        du_dt = (u[:, 1:-1, 2:] - u[:, 1:-1, :-2]) / (2.0 * self.dt)
        du_dx = (u[:, 2:, 1:-1] - u[:, :-2, 1:-1]) / (2.0 * self.dx)
        d2u_dx2 = (u[:, 2:, 1:-1] - 2.0 * u_int + u[:, :-2, 1:-1]) / (self.dx * self.dx)
        res = du_dt + u_int * du_dx - self.nu * d2u_dx2
        return res.reshape(B, 1, -1)  # (B, 1, (H-2)*(W-2))


def dirichlet_mask(H, W):
    # True on the first and last spatial rows (x = -1 and x = 1), for all times.
    mask = torch.zeros((H, W), dtype=torch.bool)
    mask[0, :] = True
    mask[-1, :] = True
    return mask.view(-1)  # (H*W,)

In [ ]:
pde_condition = BurgersCondition(H, W, sigma=1e-5).to(device)
boundary_condition = PointWiseCondition(
    torch.zeros(H * W, device=device), dirichlet_mask(H, W), sigma=1e-6).to(device)
condition = ProductCondition([pde_condition, boundary_condition]).to(device)
guidance = GuidanceMC(flow, condition, num_mc_samples=1).to(device)
samples = sampler.sample(num_samples, guidance=guidance, num_steps=1_000)
show_fields(samples.view(samples.shape[0], H, W)[:5], cmap="bwr")

## 4. RFF Weight-Space Flow

Same problem, but the diffusion runs over basis weights rather than the $N$ grid values, and the initial condition enters through a *conditioned* basis rather than a constraint. `DecoupledRFFBasis` implements pathwise conditioning (Wilson et al., 2020). The latent width becomes $R_f + M$, where the extra $M$ features carry the $M$ initial condition evaluations.

In [ ]:
basis = DecoupledRFFBasis(
    num_features=R, input_dim=dim, kernel=kernel,
    train_x=X_initial, train_y=Y_initial, noise=noise, device=device,
)
print(f"latent width R_f + M = {basis.num_features} + {basis.M} = {basis.R}")

Prior draws under the conditioned basis already respect the initial condition, but know nothing about the PDE or the boundaries.

In [ ]:
weight_flow = BasisGaussianProcessFlow(basis, coords, output_shape=(H, W), device=device)
weight_sampler = IntegratorSampler(flow=weight_flow)
uncond_weight_samples = weight_sampler.sample(num_samples, guidance=None, num_steps=1_000)
show_fields(uncond_weight_samples.view(-1, H, W)[:5], cmap="bwr")

The PDE and boundary conditions in weight space. A sampled function is $w^\top\phi(x) + m(x)$.

In [ ]:
class BurgersWeightCondition(EqualityCondition):
    """Burgers residual in weight space, using the basis's analytic derivatives."""

    def __init__(self, basis, grid, H, W, sigma=10.0):
        super().__init__(sigma=sigma)
        self.nu = 0.02  # viscosity
        sx = 0.5  # d/dx = sx * d/dcoord, since x = 2*coord - 1

        interior = torch.zeros((H, W), dtype=torch.bool)
        interior[1:-1, 1:-1] = True
        interior = interior.view(-1)  # (N,)

        self.phi = basis.phi(grid)[:, interior]  # (R, A)
        self.dphi_dx = sx * basis.compute_derivative(grid, (0,), (1,))[:, interior]
        self.dphi_dx2 = sx**2 * basis.compute_derivative(grid, (0,), (2,))[:, interior]
        self.dphi_dt = basis.compute_derivative(grid, (1,), (1,))[:, interior]

        self.mean = basis.mean(grid)[interior]  # (A,)
        self.dmean_dx = sx * basis.mean_derivative(grid, (0,), (1,))[interior]
        self.dmean_dx2 = sx**2 * basis.mean_derivative(grid, (0,), (2,))[interior]
        self.dmean_dt = basis.mean_derivative(grid, (1,), (1,))[interior]

    def residual(self, w):  # w: (B, 1, R)
        f = w @ self.phi + self.mean
        f_x = w @ self.dphi_dx + self.dmean_dx
        f_xx = w @ self.dphi_dx2 + self.dmean_dx2
        f_t = w @ self.dphi_dt + self.dmean_dt
        return f_t + f * f_x - self.nu * f_xx  # (B, 1, A)


class PointWiseWeightCondition(EqualityCondition):
    """Pin the field to a target at masked grid points, in weight space."""

    def __init__(self, target_field, mask, basis, grid, sigma=1e-6):
        super().__init__(sigma=sigma)
        self.target_field = target_field
        self.mask = mask
        self.phi = basis.phi(grid)
        self.mean = basis.mean(grid)

    def residual(self, w):  # w: (B, 1, R)
        f = w @ self.phi + self.mean
        return f[:, :, self.mask] - self.target_field[self.mask]

Sample conditioned weights; the flow maps them back to function space.

In [ ]:
weight_boundary_condition = PointWiseWeightCondition(
    torch.zeros(H * W, device=device), dirichlet_mask(H, W), basis, coords, sigma=1e-6).to(device)
weight_pde_condition = BurgersWeightCondition(basis, coords, H, W, sigma=1e-5).to(device)
weight_condition = ProductCondition([weight_boundary_condition, weight_pde_condition])
weight_guidance = GuidanceMC(weight_flow, weight_condition, num_mc_samples=1).to(device)
weight_samples = weight_sampler.sample(num_samples, guidance=weight_guidance, num_steps=1_000)
show_fields(weight_samples, cmap="bwr")

## 5. Cost

Both flows are run at increasing grid size $N = H \times W$ ($W$ fixed, $H$ growing), with `num_steps=250`.

* **Function space** pays an $\mathcal{O}(N^3)$ Cholesky once, stores the $(N, N)$ factor, and then pays an $\mathcal{O}(N^2)$ matmul per step.
* **Weight space** keeps a state of width $R_f + M$, so its cost is linear in $N$.


In [ ]:
BENCH_STEPS = 250    # per-step cost is what scales; the runs above use 1_000
BENCH_H = [50, 100, 200, 400, 800, 1600]
FUNC_BUDGET = 30.0   # seconds; skip function space once its next setup would exceed this


def _sync():
    if device == "cuda":
        torch.cuda.synchronize()  # time the work, not the asynchronous launch


def run_function_space(H, W):
    crd = build_grid(H, W, device)
    _sync(); t0 = time.perf_counter()
    mu, kk = gp_posterior(kernel, X_initial, Y_initial, noise=noise)
    fl = GaussianProcessFlow(kk, mu, crd, device=device)  # O(N^3) Cholesky, O(N^2) storage
    _sync(); t_setup = time.perf_counter() - t0

    cond = ProductCondition([
        BurgersCondition(H, W, sigma=1e-5),
        PointWiseCondition(torch.zeros(H * W, device=device), dirichlet_mask(H, W), sigma=1e-6),
    ]).to(device)
    gd = GuidanceMC(fl, cond, num_mc_samples=1).to(device)

    _sync(); t0 = time.perf_counter()
    out = IntegratorSampler(flow=fl).sample(num_samples, guidance=gd, num_steps=BENCH_STEPS, verbose=False)
    _sync()
    return t_setup, time.perf_counter() - t0, out.reshape(num_samples, 1, H * W)


def run_weight_space(H, W):
    crd = build_grid(H, W, device)
    _sync(); t0 = time.perf_counter()
    bs = DecoupledRFFBasis(num_features=R, input_dim=dim, kernel=kernel,
                           train_x=X_initial, train_y=Y_initial, noise=noise, device=device)
    fl = BasisGaussianProcessFlow(bs, crd, output_shape=(H, W), device=device)
    _sync(); t_setup = time.perf_counter() - t0

    cond = ProductCondition([
        BurgersWeightCondition(bs, crd, H, W, sigma=1e-5),
        PointWiseWeightCondition(torch.zeros(H * W, device=device), dirichlet_mask(H, W),
                                 bs, crd, sigma=1e-6),
    ]).to(device)
    gd = GuidanceMC(fl, cond, num_mc_samples=1).to(device)

    _sync(); t0 = time.perf_counter()
    out = IntegratorSampler(flow=fl).sample(num_samples, guidance=gd, num_steps=BENCH_STEPS, verbose=False)
    _sync()
    return t_setup, time.perf_counter() - t0, out.reshape(num_samples, 1, H * W)


results = {"func": [], "rff": []}
func_alive = True
for H_b in BENCH_H:
    N_b = H_b * W
    for name, runner in (("func", run_function_space), ("rff", run_weight_space)):
        if name == "func" and not func_alive:
            continue
        try:
            t_setup, t_sample, out = runner(H_b, W)
        except (RuntimeError, MemoryError) as err:  # the (N, N) factor no longer fits
            print(f"N = {N_b:6d}  {name:4s}  not run: {type(err).__name__}")
            func_alive = False
            continue
        # Score both with the same finite-difference residual.
        resid = BurgersCondition(H_b, W).to(device).residual(out).pow(2).mean().sqrt().item()
        results[name].append((N_b, t_setup, t_sample, resid))
        print(f"N = {N_b:6d}  {name:4s}  setup {t_setup:7.2f}s   "
              f"sample {t_sample:6.2f}s   residual {resid:.3e}")
        del out
        if device == "cuda":
            torch.cuda.empty_cache()
    # The Cholesky is O(N^3), so doubling H costs ~8x.
    if func_alive and results["func"] and results["func"][-1][1] * 8 > FUNC_BUDGET:
        print(f"N = {N_b:6d}  func  stopping: next setup would exceed {FUNC_BUDGET:.0f}s")
        func_alive = False

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for name, label, col, power in (("func", "function space", "tab:orange", 3),
                                ("rff", "RFF weight space", "tab:blue", 1)):
    Ns, setup, sample, _ = (list(v) for v in zip(*results[name]))
    total = [s + p for s, p in zip(setup, sample)]
    ax.plot(Ns, total, marker="o", color=col, label=label)
    ax.plot(Ns, [total[-1] * (n / Ns[-1]) ** power for n in Ns], ls="--", lw=1.2,
            color=col, alpha=0.55, label=f"$O(N^{power})$, {label}" if power > 1 else f"$O(N)$, {label}")

ax.set(xscale="log", yscale="log", xlabel="N = H x W", ylabel="wall time (s)",
       title=f"Cost of one run ({BENCH_STEPS} steps, {num_samples} samples)")
ax.grid(True, which="both", ls="--", alpha=0.4)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

func_N, func_t = results["func"][-1][0], results["func"][-1][1] + results["func"][-1][2]
rff_t = next(s + p for n, s, p, _ in results["rff"] if n == func_N)
print(f"\nAt N = {func_N}, weight space is {func_t / rff_t:.0f}x faster for the same residual band.")
print(f"Function space stopped at N = {func_N}; weight space reached N = {results['rff'][-1][0]} "
      f"in {results['rff'][-1][1] + results['rff'][-1][2]:.1f}s.")